# Exploratory analysis comparing 2024 marathon prep to 2025 marathon prep
## Identify / Compare:
[]average pace
[]weekly average mileage
[]monthly average mileage from july to October
[]anything interesting about heart rate data

## Potential Questions
1. which training period had more prep
2. training plan didn't change a whole lot - why were times so different
3. maybe incorporate some sort of regression analysis

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

activities = pd.read_csv('../data/activities.csv')
activities.head()

,Activity ID,Activity Date,Activity Name,Activity Type,Activity Description,Elapsed Time,Distance,Max Heart Rate,Relative Effort,Commute,...,Recovery,With Pet,Competition,Long Run,For a Cause,With Kid,Downhill Distance,Total Sets,Total Reps,Media
0,1367203136,"Jan 21, 2018, 1:26:02 AM",Evening Run,Run,NaN,769,2.74,209.0,45.0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1367203273,"Jan 19, 2018, 10:55:21 PM",Afternoon Run,Run,NaN,6569,10.37,171.0,30.0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1371326189,"Jan 23, 2018, 12:10:09 PM",Morning Run,Run,NaN,2867,7.36,192.0,36.0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1372867134,"Jan 24, 2018, 11:30:54 AM",Morning Run,Run,NaN,4738,8.19,175.0,25.0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1375484414,"Jan 26, 2018, 12:52:39 AM",Evening Run,Run,NaN,1011,3.27,176.0,31.0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Try to get a feel for the df
activities.info()

<class 'pandas.DataFrame'>
RangeIndex: 639 entries, 0 to 638
Columns: 103 entries, Activity ID to Media
dtypes: bool(1), float64(93), int64(2), str(7)
memory usage: 510.0 KB


In [25]:
# Yikes - 103 columns is a lot
list(activities.columns)

['Activity ID',
 'Activity Date',
 'Activity Name',
 'Activity Type',
 'Activity Description',
 'Elapsed Time',
 'Distance',
 'Max Heart Rate',
 'Relative Effort',
 'Commute',
 'Activity Private Note',
 'Activity Gear',
 'Filename',
 'Athlete Weight',
 'Bike Weight',
 'Elapsed Time.1',
 'Moving Time',
 'Distance.1',
 'Max Speed',
 'Average Speed',
 'Elevation Gain',
 'Elevation Loss',
 'Elevation Low',
 'Elevation High',
 'Max Grade',
 'Average Grade',
 'Average Positive Grade',
 'Average Negative Grade',
 'Max Cadence',
 'Average Cadence',
 'Max Heart Rate.1',
 'Average Heart Rate',
 'Max Watts',
 'Average Watts',
 'Calories',
 'Max Temperature',
 'Average Temperature',
 'Relative Effort.1',
 'Total Work',
 'Number of Runs',
 'Uphill Time',
 'Downhill Time',
 'Other Time',
 'Perceived Exertion',
 'Type',
 'Start Time',
 'Weighted Average Power',
 'Power Count',
 'Prefer Perceived Exertion',
 'Perceived Relative Effort',
 'Commute.1',
 'Total Weight Lifted',
 'From Upload',
 'Grade Adj

In [26]:
# Lets filter to only Run activity types
isRun = activities['Activity Type'] == 'Run'
activities = activities[isRun]
activities.info()

<class 'pandas.DataFrame'>
Index: 592 entries, 0 to 638
Columns: 103 entries, Activity ID to Media
dtypes: bool(1), float64(93), int64(2), str(7)
memory usage: 477.0 KB


In [27]:
# Now let's trim down to only relevant columns
activities = activities[['Activity Type', 'Distance', 'Moving Time', 'Average Heart Rate', 'Activity Date', 'Elevation Gain', 'Activity Description']]
activities.head()

,Activity Type,Distance,Moving Time,Average Heart Rate,Activity Date,Elevation Gain,Activity Description
0,Run,2.74,735.0,180.0,"Jan 21, 2018, 1:26:02 AM",20.0,NaN
1,Run,10.37,6528.0,115.0,"Jan 19, 2018, 10:55:21 PM",0.0,NaN
2,Run,7.36,2343.0,143.0,"Jan 23, 2018, 12:10:09 PM",0.0,NaN
3,Run,8.19,4628.0,123.0,"Jan 24, 2018, 11:30:54 AM",0.0,NaN
4,Run,3.27,899.0,158.0,"Jan 26, 2018, 12:52:39 AM",21.9,NaN


In [28]:
# Let's check data types before calculations
activities.info()

<class 'pandas.DataFrame'>
Index: 592 entries, 0 to 638
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Activity Type         592 non-null    str    
 1   Distance              592 non-null    float64
 2   Moving Time           592 non-null    float64
 3   Average Heart Rate    555 non-null    float64
 4   Activity Date         592 non-null    str    
 5   Elevation Gain        588 non-null    float64
 6   Activity Description  202 non-null    str    
dtypes: float64(4), str(3)
memory usage: 37.0 KB


In [29]:
# Yikes - need to convert Activity Date to a date datatype
activities['Activity Date'] = pd.to_datetime(activities['Activity Date'])
activities.info()

<class 'pandas.DataFrame'>
Index: 592 entries, 0 to 638
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Activity Type         592 non-null    str           
 1   Distance              592 non-null    float64       
 2   Moving Time           592 non-null    float64       
 3   Average Heart Rate    555 non-null    float64       
 4   Activity Date         592 non-null    datetime64[us]
 5   Elevation Gain        588 non-null    float64       
 6   Activity Description  202 non-null    str           
dtypes: datetime64[us](1), float64(4), str(2)
memory usage: 37.0 KB


In [30]:
activities.head()

,Activity Type,Distance,Moving Time,Average Heart Rate,Activity Date,Elevation Gain,Activity Description
0,Run,2.74,735.0,180.0,2018-01-21 01:26:02,20.0,NaN
1,Run,10.37,6528.0,115.0,2018-01-19 22:55:21,0.0,NaN
2,Run,7.36,2343.0,143.0,2018-01-23 12:10:09,0.0,NaN
3,Run,8.19,4628.0,123.0,2018-01-24 11:30:54,0.0,NaN
4,Run,3.27,899.0,158.0,2018-01-26 00:52:39,21.9,NaN


In [31]:
# Okay we're getting there - need to convert moving time from seconds to minutes & generate pace per mile
activities['Moving Time'] = activities['Moving Time'] / 60
activities.sort_values(by='Distance')

,Activity Type,Distance,Moving Time,Average Heart Rate,Activity Date,Elevation Gain,Activity Description
410,Run,0.00,105.000000,NaN,2023-07-27 11:00:30,NaN,Deadlift: 260 (leave me alone)\nBall throw: 10...
33,Run,0.23,1.150000,NaN,2018-06-06 15:10:25,0.0,NaN
18,Run,0.33,1.266667,NaN,2018-05-02 23:55:27,0.0,NaN
162,Run,0.39,1.316667,149.0,2021-05-04 16:54:09,2.0,3/5
161,Run,0.39,1.366667,148.0,2021-05-04 16:49:53,1.0,2/5
...,...,...,...,...,...,...,...
14,Run,42.05,414.000000,140.0,2018-03-25 13:45:46,636.3,NaN
115,Run,42.22,270.016667,150.0,2020-12-05 15:20:08,83.0,hit a phat wall at mile 18 that i couldn’t bre...
519,Run,42.40,233.216667,164.0,2024-10-13 07:45:36,98.0,I’m tired
618,Run,42.87,226.650000,161.0,2025-10-05 08:31:28,201.0,do u even bonk


In [32]:
# Welp I guess the distance is in kilometers
activities['DistanceInMiles'] = activities['Distance'] * .621371
activities['Pace'] = activities['Moving Time'] / activities['DistanceInMiles']
activities.head()

,Activity Type,Distance,Moving Time,Average Heart Rate,Activity Date,Elevation Gain,Activity Description,DistanceInMiles,Pace
0,Run,2.74,12.250000,180.0,2018-01-21 01:26:02,20.0,NaN,1.702557,7.195062
1,Run,10.37,108.800000,115.0,2018-01-19 22:55:21,0.0,NaN,6.443617,16.884926
2,Run,7.36,39.050000,143.0,2018-01-23 12:10:09,0.0,NaN,4.573291,8.538710
3,Run,8.19,77.133333,123.0,2018-01-24 11:30:54,0.0,NaN,5.089028,15.156789
4,Run,3.27,14.983333,158.0,2018-01-26 00:52:39,21.9,NaN,2.031883,7.374112
